# N06 — Điền NULL cột serum_sodium
**Võ Minh Huy — 2311110117**

Notebook chạy độc lập bằng **Run All**. Các ô đầu dùng lại bước chuẩn bị và N01–N05 đã hoàn thành để tạo đầu vào sau serum_creatinine giống SQL. Không cần chạy notebook khác trước. CSV before được giữ nguyên. Chụp code và tổng kết ở mục 3, nhật ký ở mục 4.

## 1. Tạo lại đầu vào sau N05
Đây là code các bước cũ, chưa phải nội dung mới của N06.

In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from collections import Counter
import csv
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_csv = thu_muc / "VoMinhHuy_Heart Failure Prediction_before.csv"
with file_csv.open(encoding="utf-8-sig", newline="") as f:
    du_lieu_goc = list(csv.DictReader(f))

def doc_so(gia_tri, cot):
    chuoi = gia_tri.strip()
    if cot == "sex":
        chuoi = {"m":"1", "male":"1", "f":"0", "female":"0"}.get(chuoi.lower(), chuoi)
    try:
        return Decimal(chuoi).quantize(Decimal("0.000001"))
    except InvalidOperation:
        return None

ho_so, da_gap = [], set()
for dong in du_lieu_goc:
    if not any(v.strip() for v in dong.values()):
        continue
    ban_ghi = {cot: doc_so(v, cot) for cot, v in dong.items()}
    khoa = tuple(ban_ghi.values())
    if khoa not in da_gap:
        da_gap.add(khoa)
        ho_so.append(ban_ghi)
df = pd.DataFrame(ho_so)
df.insert(0, "MaHoSo", range(1, len(df)+1))
mau_cho = df.loc[df["sex"].isin([0, 1])].copy()
pd.DataFrame({"SoHoSo":[len(df)], "SoSexThieuTruocDien":[df["sex"].isna().sum()]})

,SoHoSo,SoSexThieuTruocDien
0,299,20


In [2]:
cac_nhom = [
    ["anaemia", "diabetes", "smoking", "high_blood_pressure"],
    ["anaemia", "diabetes"], ["diabetes"], []
]
nhat_ky = []
for i in df.index[df["sex"].isna()]:
    ket_qua = {"MaHoSo":df.at[i, "MaHoSo"],
               **{c:df.at[i,c] for c in cac_nhom[0]},
               "CapNhom":None, "SoMauCho":0, "GiaTriDien":None}
    for cap, cac_cot in enumerate(cac_nhom, start=1):
        nhom = mau_cho
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(df.at[i,cot])]
        if len(nhom) < 5:
            continue
        tan_suat = Counter(nhom["sex"])
        mode = min(v for v, n in tan_suat.items() if n == max(tan_suat.values()))
        df.at[i, "sex"] = mode
        ket_qua.update(CapNhom=cap, SoMauCho=len(nhom), GiaTriDien=mode)
        break
    nhat_ky.append(ket_qua)
nhat_ky = pd.DataFrame(nhat_ky)
tong_ket = pd.DataFrame({"SoHoSo":[len(df)],
    "SoSexThieuTruocDien":[len(nhat_ky)],
    "SoODaDien":[nhat_ky["GiaTriDien"].notna().sum()],
    "SoSexThieuSauDien":[df["sex"].isna().sum()]})
tong_ket

,SoHoSo,SoSexThieuTruocDien,SoODaDien,SoSexThieuSauDien
0,299,20,20,0


In [3]:
from decimal import ROUND_HALF_UP
from statistics import median

# df là kết quả sau N01; ho_so giữ dữ liệu quan sát trước N01.
age_work = df.copy(deep=True)
truoc_n01 = pd.DataFrame(ho_so)
tuoi_nhieu = [Decimal("-12"), Decimal("250")]
mat_na_can_dien = age_work["age"].isna() | age_work["age"].isin(tuoi_nhieu)
age_targets = age_work.loc[mat_na_can_dien, ["MaHoSo", "age", "sex", "diabetes"]].copy()
age_donors = truoc_n01.loc[
    truoc_n01["age"].notna() & ~truoc_n01["age"].isin(tuoi_nhieu),
    ["age", "sex", "diabetes"]
].copy()
age_work.loc[age_work["age"].isin(tuoi_nhieu), "age"] = None

pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien
0,299,21,2,23


In [4]:
cac_nhom_age = [["sex", "diabetes"], ["sex"], []]
dong_nhat_ky_age = []
for i, r in age_targets.iterrows():
    ket_qua_age = {"MaHoSo":r["MaHoSo"], "TuoiTruoc":r["age"],
        "sex":r["sex"], "diabetes":r["diabetes"],
        "LyDo":"Tuổi bị làm nhiễu" if r["age"] in tuoi_nhieu else "NULL hoặc sai kiểu",
        "CapNhom":None, "SoMauCho":0, "MedianNhom":None, "TuoiDien":None}
    for cap, cac_cot in enumerate(cac_nhom_age, start=1):
        nhom = age_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom["age"].tolist())
        tuoi_dien = median_nhom.quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)
        age_work.at[i, "age"] = tuoi_dien
        ket_qua_age.update(CapNhom=cap, SoMauCho=len(nhom),
                          MedianNhom=median_nhom, TuoiDien=tuoi_dien)
        break
    dong_nhat_ky_age.append(ket_qua_age)
nhat_ky_age = pd.DataFrame(dong_nhat_ky_age)
tong_ket_age = pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)],
    "SoODaDien":[nhat_ky_age["TuoiDien"].notna().sum()],
    "SoAgeThieuSauDien":[age_work["age"].isna().sum()]})
tong_ket_age

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien,SoODaDien,SoAgeThieuSauDien
0,299,21,2,23,23,0


In [5]:
def nhom_tuoi(tuoi):
    if pd.isna(tuoi) or tuoi in tuoi_nhieu:
        return None
    return "A1" if tuoi < 50 else "A2" if tuoi < 60 else "A3" if tuoi < 70 else "A4"

cot_cpk = "creatinine_phosphokinase"
cpk_nhieu = Decimal("99999")
cpk_work = age_work.copy(deep=True)
cpk_work["age_group"] = cpk_work["age"].map(nhom_tuoi)
mat_na_cpk = cpk_work[cot_cpk].isna() | cpk_work[cot_cpk].eq(cpk_nhieu)
cpk_targets = cpk_work.loc[mat_na_cpk,
    ["MaHoSo", cot_cpk, "age", "age_group", "anaemia"]].copy()
cpk_targets = cpk_targets.rename(columns={cot_cpk:"CPKTruoc"})

# Mẫu cho cố định từ dữ liệu quan sát trước N01, N02.
cpk_donors = truoc_n01.loc[
    truoc_n01[cot_cpk].notna() & ~truoc_n01[cot_cpk].eq(cpk_nhieu),
    [cot_cpk, "age", "anaemia"]].copy()
cpk_donors["age_group"] = cpk_donors["age"].map(nhom_tuoi)
cpk_work.loc[cpk_work[cot_cpk].eq(cpk_nhieu), cot_cpk] = None

pd.DataFrame({"SoHoSo":[len(cpk_work)],
    "SoNullHoacSaiKieu":[cpk_targets["CPKTruoc"].isna().sum()],
    "SoCPKNhieu":[cpk_targets["CPKTruoc"].eq(cpk_nhieu).sum()],
    "SoOCanDien":[len(cpk_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoCPKNhieu,SoOCanDien
0,299,20,1,21


In [6]:
cac_nhom_cpk = [["age_group", "anaemia"], ["age_group"], []]
dong_nhat_ky_cpk = []
for i, r in cpk_targets.iterrows():
    ket_qua_cpk = r.to_dict()
    ket_qua_cpk.update(
        LyDo="CPK bị làm nhiễu" if r["CPKTruoc"] == cpk_nhieu else "NULL hoặc sai kiểu",
        CapNhom=None, SoMauCho=0, MedianNhom=None, CPKDien=None)
    for cap, cac_cot in enumerate(cac_nhom_cpk, start=1):
        nhom = cpk_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom[cot_cpk].tolist())
        cpk_dien = median_nhom.quantize(Decimal("1"), rounding=ROUND_HALF_UP)
        cpk_work.at[i, cot_cpk] = cpk_dien
        ket_qua_cpk.update(CapNhom=cap, SoMauCho=len(nhom),
                          MedianNhom=median_nhom, CPKDien=cpk_dien)
        break
    dong_nhat_ky_cpk.append(ket_qua_cpk)
nhat_ky_cpk = pd.DataFrame(dong_nhat_ky_cpk)
tong_ket_cpk = pd.DataFrame({"SoHoSo":[len(cpk_work)],
    "SoNullHoacSaiKieu":[cpk_targets["CPKTruoc"].isna().sum()],
    "SoCPKNhieu":[cpk_targets["CPKTruoc"].eq(cpk_nhieu).sum()],
    "SoOCanDien":[len(cpk_targets)],
    "SoODaDien":[nhat_ky_cpk["CPKDien"].notna().sum()],
    "SoCPKThieuSauDien":[cpk_work[cot_cpk].isna().sum()]})
tong_ket_cpk

,SoHoSo,SoNullHoacSaiKieu,SoCPKNhieu,SoOCanDien,SoODaDien,SoCPKThieuSauDien
0,299,20,1,21,21,0


In [7]:
from decimal import localcontext

platelets_work = cpk_work.copy(deep=True)
mat_na_platelets = platelets_work["platelets"].isna()
platelets_targets = platelets_work.loc[mat_na_platelets,
    ["MaHoSo", "platelets", "sex", "anaemia"]].copy()
platelets_targets = platelets_targets.rename(columns={"platelets":"PlateletsTruoc"})
platelets_donors = truoc_n01.loc[truoc_n01["platelets"].notna(),
    ["platelets", "sex", "anaemia"]].copy()

pd.DataFrame({"SoHoSo":[len(platelets_work)],
    "SoPlateletsThieuTruocDien":[len(platelets_targets)]})

,SoHoSo,SoPlateletsThieuTruocDien
0,299,20


In [8]:
cac_nhom_platelets = [["sex", "anaemia"], ["sex"], []]
dong_nhat_ky_platelets = []
for i, r in platelets_targets.iterrows():
    ket_qua_platelets = r.to_dict()
    ket_qua_platelets.update(CapNhom=None, SoMauCho=0,
                            MeanNhom=None, PlateletsDien=None)
    for cap, cac_cot in enumerate(cac_nhom_platelets, start=1):
        nhom = platelets_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        with localcontext() as ctx:
            ctx.prec = 38
            mean_nhom = (sum(nhom["platelets"], Decimal(0)) / len(nhom)).quantize(
                Decimal("0.0000000001"), rounding=ROUND_HALF_UP)
            gia_tri_dien = mean_nhom.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        platelets_work.at[i, "platelets"] = gia_tri_dien
        ket_qua_platelets.update(CapNhom=cap, SoMauCho=len(nhom),
                                MeanNhom=mean_nhom, PlateletsDien=gia_tri_dien)
        break
    dong_nhat_ky_platelets.append(ket_qua_platelets)
nhat_ky_platelets = pd.DataFrame(dong_nhat_ky_platelets)
tong_ket_platelets = pd.DataFrame({"SoHoSo":[len(platelets_work)],
    "SoPlateletsThieuTruocDien":[len(platelets_targets)],
    "SoODaDien":[nhat_ky_platelets["PlateletsDien"].notna().sum()],
    "SoPlateletsThieuSauDien":[platelets_work["platelets"].isna().sum()]})
tong_ket_platelets

,SoHoSo,SoPlateletsThieuTruocDien,SoODaDien,SoPlateletsThieuSauDien
0,299,20,20,0


In [9]:
cot_creatinine = "serum_creatinine"
creatinine_work = platelets_work.copy(deep=True)
mat_na_creatinine = creatinine_work[cot_creatinine].isna()
creatinine_targets = creatinine_work.loc[mat_na_creatinine,
    ["MaHoSo", cot_creatinine, "age", "age_group", "high_blood_pressure"]].copy()
creatinine_targets = creatinine_targets.rename(columns={cot_creatinine:"CreatinineTruoc"})
creatinine_donors = truoc_n01.loc[truoc_n01[cot_creatinine].notna(),
    [cot_creatinine, "age", "high_blood_pressure"]].copy()
creatinine_donors["age_group"] = creatinine_donors["age"].map(nhom_tuoi)

pd.DataFrame({"SoHoSo":[len(creatinine_work)],
    "SoCreatinineThieuTruocDien":[len(creatinine_targets)]})

,SoHoSo,SoCreatinineThieuTruocDien
0,299,20


In [10]:
cac_nhom_creatinine = [["age_group", "high_blood_pressure"], ["age_group"], []]
dong_nhat_ky_creatinine = []
for i, r in creatinine_targets.iterrows():
    ket_qua_creatinine = r.to_dict()
    ket_qua_creatinine.update(CapNhom=None, SoMauCho=0,
                             MedianNhom=None, CreatinineDien=None)
    for cap, cac_cot in enumerate(cac_nhom_creatinine, start=1):
        nhom = creatinine_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom[cot_creatinine].tolist())
        gia_tri_dien = median_nhom.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        creatinine_work.at[i, cot_creatinine] = gia_tri_dien
        ket_qua_creatinine.update(CapNhom=cap, SoMauCho=len(nhom),
                                 MedianNhom=median_nhom, CreatinineDien=gia_tri_dien)
        break
    dong_nhat_ky_creatinine.append(ket_qua_creatinine)
nhat_ky_creatinine = pd.DataFrame(dong_nhat_ky_creatinine)
tong_ket_creatinine = pd.DataFrame({"SoHoSo":[len(creatinine_work)],
    "SoCreatinineThieuTruocDien":[len(creatinine_targets)],
    "SoODaDien":[nhat_ky_creatinine["CreatinineDien"].notna().sum()],
    "SoCreatinineThieuSauDien":[creatinine_work[cot_creatinine].isna().sum()]})
tong_ket_creatinine

,SoHoSo,SoCreatinineThieuTruocDien,SoODaDien,SoCreatinineThieuSauDien
0,299,20,20,0


## 2. Chuẩn bị natri và mẫu cho cố định
Có 20 ô thiếu và 1 giá trị nhiễu 999 cần xử lý. Hồ sơ nhận dùng nhóm tuổi sau N02. Mẫu cho dùng natri và nhóm tuổi quan sát trước mọi bước điền; loại 999. Tuổi thiếu/nhiễu không tạo khóa nhóm, nhưng natri hợp lệ vẫn dùng ở cấp toàn cột.

In [11]:
cot_sodium = "serum_sodium"
sodium_nhieu = Decimal("999")
sodium_work = creatinine_work.copy(deep=True)
mat_na_sodium = sodium_work[cot_sodium].isna() | sodium_work[cot_sodium].eq(sodium_nhieu)
sodium_targets = sodium_work.loc[mat_na_sodium,
    ["MaHoSo", cot_sodium, "age", "age_group", "diabetes"]].copy()
sodium_targets = sodium_targets.rename(columns={cot_sodium:"SodiumTruoc"})
sodium_donors = truoc_n01.loc[
    truoc_n01[cot_sodium].notna() & ~truoc_n01[cot_sodium].eq(sodium_nhieu),
    [cot_sodium, "age", "diabetes"]].copy()
sodium_donors["age_group"] = sodium_donors["age"].map(nhom_tuoi)
sodium_work.loc[sodium_work[cot_sodium].eq(sodium_nhieu), cot_sodium] = None

pd.DataFrame({"SoHoSo":[len(sodium_work)],
    "SoNullHoacSaiKieu":[sodium_targets["SodiumTruoc"].isna().sum()],
    "SoSodiumNhieu":[sodium_targets["SodiumTruoc"].eq(sodium_nhieu).sum()],
    "SoOCanDien":[len(sodium_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoSodiumNhieu,SoOCanDien
0,299,20,1,21


## 3. Điền mode theo nhóm và tổng kết
Thử lần lượt: nhóm tuổi và diabetes → nhóm tuổi → toàn cột. Dùng nhóm đầu có ít nhất 5 mẫu; không đủ ở mọi cấp thì giữ NULL. Mode là giá trị có tần suất cao nhất; đồng mode chọn số nhỏ nhất. Chỉ giá trị mới điền được làm tròn số nguyên.

In [12]:
cac_nhom_sodium = [["age_group", "diabetes"], ["age_group"], []]
dong_nhat_ky_sodium = []
for i, r in sodium_targets.iterrows():
    ket_qua_sodium = r.to_dict()
    ket_qua_sodium.update(
        LyDo="Natri bị làm nhiễu" if r["SodiumTruoc"] == sodium_nhieu else "NULL hoặc sai kiểu",
        CapNhom=None, SoMauCho=0, TanSuatMode=None, ModeNhom=None, SodiumDien=None)
    for cap, cac_cot in enumerate(cac_nhom_sodium, start=1):
        nhom = sodium_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        tan_suat = Counter(nhom[cot_sodium])
        tan_suat_mode = max(tan_suat.values())
        mode_nhom = min(v for v, n in tan_suat.items() if n == tan_suat_mode)
        gia_tri_dien = mode_nhom.quantize(Decimal("1"), rounding=ROUND_HALF_UP)
        sodium_work.at[i, cot_sodium] = gia_tri_dien
        ket_qua_sodium.update(CapNhom=cap, SoMauCho=len(nhom),
                             TanSuatMode=tan_suat_mode, ModeNhom=mode_nhom,
                             SodiumDien=gia_tri_dien)
        break
    dong_nhat_ky_sodium.append(ket_qua_sodium)
nhat_ky_sodium = pd.DataFrame(dong_nhat_ky_sodium)
tong_ket_sodium = pd.DataFrame({"SoHoSo":[len(sodium_work)],
    "SoNullHoacSaiKieu":[sodium_targets["SodiumTruoc"].isna().sum()],
    "SoSodiumNhieu":[sodium_targets["SodiumTruoc"].eq(sodium_nhieu).sum()],
    "SoOCanDien":[len(sodium_targets)],
    "SoODaDien":[nhat_ky_sodium["SodiumDien"].notna().sum()],
    "SoSodiumThieuSauDien":[sodium_work[cot_sodium].isna().sum()]})
tong_ket_sodium

,SoHoSo,SoNullHoacSaiKieu,SoSodiumNhieu,SoOCanDien,SoODaDien,SoSodiumThieuSauDien
0,299,20,1,21,21,0


## 4. Nhật ký 21 ô serum_sodium
Chụp cả bảng: natri trước xử lý, tuổi, nhóm tuổi, diabetes, lý do, cấp nhóm, số mẫu cho, tần suất mode, mode và giá trị điền. MaHoSo là số thứ tự nội bộ, có thể khác SQL.

In [13]:
nhat_ky_sodium

,MaHoSo,SodiumTruoc,age,age_group,diabetes,LyDo,CapNhom,SoMauCho,TanSuatMode,ModeNhom,SodiumDien
0,6,None,60.0,A3,0.000000,NULL hoặc sai kiểu,1,37,5,138.000000,138
1,19,999.000000,70.000000,A4,0.000000,Natri bị làm nhiễu,1,44,9,134.000000,134
2,28,None,70.000000,A4,1.000000,NULL hoặc sai kiểu,1,19,6,134.000000,134
3,38,None,82.000000,A4,1.000000,NULL hoặc sai kiểu,1,19,6,134.000000,134
4,50,None,57.000000,A2,0.000000,NULL hoặc sai kiểu,1,39,8,137.000000,137
5,63,None,55.000000,A2,0.000000,NULL hoặc sai kiểu,1,39,8,137.000000,137
6,77,None,70.000000,A4,0.000000,NULL hoặc sai kiểu,1,44,9,134.000000,134
7,90,None,57.000000,A2,0.000000,NULL hoặc sai kiểu,1,39,8,137.000000,137
8,110,None,60.8,A3,1.000000,NULL hoặc sai kiểu,1,48,10,136.000000,136
9,118,None,85.000000,A4,0.000000,NULL hoặc sai kiểu,1,44,9,134.000000,134


## 5. Lưu kết quả riêng N06
Lưu tổng kết, nhật ký và bản làm việc trung gian. Chưa tạo dữ liệu after cuối.

In [14]:
thu_muc_sodium = thu_muc / "ket_qua" / "10_dien_serum_sodium"
thu_muc_sodium.mkdir(parents=True, exist_ok=True)
tong_ket_sodium.to_csv(thu_muc_sodium / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhat_ky_sodium.to_csv(thu_muc_sodium / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
sodium_work.to_csv(thu_muc_sodium / "ban_lam_viec_sau_sodium_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả riêng N06; CSV before được giữ nguyên."]})

,TrangThai
0,Đã lưu kết quả riêng N06; CSV before được giữ ...
